In [1]:
import torch
import torchvision
from transformers import AutoProcessor, AutoModelForTokenClassification

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

c:\Users\arunz\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.4.1+cu118
Torchvision: 0.19.1+cu118
CUDA available: True
GPU: NVIDIA GeForce GTX 1050


In [ ]:
#Install specific versions of PyTorch and related libraries to ensure compatibility with CUDA 11.8
# %pip install --upgrade --force-reinstall torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download-r2.pytorch.org/whl/cu118/torch-2.4.1%2Bcu118-cp38-cp38-win_amd64.whl (2695.5 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.19.1%2Bcu118-cp38-cp38-win_amd64.whl (5.0 MB)
  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.4.1%2Bcu118-cp38-cp38-win_amd64.whl (4.0 MB)
  Using cached networkx-3.1-py3-none-any.whl (2.1 MB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Using cached filelock-3.16.1-py3-none-any.whl (16 kB)
  Using cached sympy-1.13.3-py3-none-any.whl (6.2 MB)
  Using cached typing_extensions-4.13.2-py3-none-any.whl (45 kB)
  Using cached pillow-10.4.0-cp38-cp38-win_amd64.whl (2.6 MB)
  Using cached numpy-1.24.4-cp38-cp38-win_amd64.whl (14.9 MB)
  Using cached https://download.pytorch.org/whl/MarkupSafe-2.1.5-cp38-cp38-win_amd64.whl (17 kB)
  

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\arunz\\AppData\\Local\\Programs\\Python\\Python38\\Lib\\site-packages\\~arkupsafe\\_speedups.cp38-win_amd64.pyd'
Consider using the `--user` option or check the permissions.

You should consider upgrading via the 'c:\Users\arunz\AppData\Local\Programs\Python\Python38\python.exe -m pip install --upgrade pip' command.


In [1]:
import torch
import torchvision
from transformers import AutoProcessor, AutoModelForTokenClassification

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

c:\Users\arunz\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.4.1+cu118
Torchvision: 0.19.1+cu118
CUDA available: True
GPU: NVIDIA GeForce GTX 1050


In [2]:
# Install Tesseract-OCR engine and PyMuPDF (if not already installed in Colab environment)
!sudo apt update
!sudo apt install tesseract-ocr
!pip install PyMuPDF pytesseract Pillow

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import json # Useful for outputting results
import re # Useful for any potential regex fallback or processing

# Configure pytesseract to use the installed tesseract-ocr engine
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'

# Helper function to convert PDF page to PIL Image
def convert_pdf_page_to_image(input_path, page_num=0):
    doc = fitz.open(input_path)
    page = doc.load_page(page_num)
    pix = page.get_pixmap()
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    doc.close() # Close the document after processing
    return img

print("Core OCR and PDF processing libraries installed and helper functions defined.")

Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
You should consider upgrading via the 'c:\users\arunz\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


Core OCR and PDF processing libraries installed and helper functions defined.


## Information Extraction using LayoutLMv3 (Conceptual Example)

LayoutLMv3 is a multi-modal (text, image, layout) model for document understanding. It excels at tasks like form understanding, where the visual layout of the text is crucial for meaning. This approach typically involves:
1.  **Installing Libraries**: `transformers` for the model, `datasets` for data handling, `Pillow` for image processing.
2.  **PDF to Image Conversion**: Convert each page of the PDF into an image.
3.  **Text and Layout Extraction**: For each image, extract text, words, and their bounding box coordinates. This step is similar to OCR but focuses on word-level bounding boxes.
4.  **Tokenization and Model Input**: Prepare the extracted text and bounding boxes into a format suitable for LayoutLMv3.
5.  **Model Inference**: Use a pre-trained LayoutLMv3 model (e.g., for token classification or question answering) to identify and extract the desired entities.

This example will show the basic setup and a conceptual inference step for form understanding. For production-level accuracy, fine-tuning on a custom dataset of your forms is often recommended.

In [3]:
# Install necessary libraries for LayoutLMv3
!pip install transformers accelerate datasets
!pip install Pillow # Already installed but good to ensure

import fitz # PyMuPDF for PDF to image conversion
from PIL import Image
from transformers import AutoProcessor, AutoModelForTokenClassification
import torch

# Helper function to convert PDF page to PIL Image
def convert_pdf_page_to_image(pdf_path, page_num):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_num)
    pix = page.get_pixmap()
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return img

print("Libraries installed and helper function defined for LayoutLMv3.")

You should consider upgrading via the 'c:\users\arunz\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


Libraries installed and helper function defined for LayoutLMv3.


ERROR: Invalid requirement: '#'
You should consider upgrading via the 'c:\users\arunz\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


In [4]:
from transformers import AutoProcessor, AutoModelForTokenClassification

# --- Load pre-trained LayoutLMv3 model and processor ---
# Using a pre-trained model for document understanding (e.g., token classification)
# For specific form extraction, a fine-tuned model would be more accurate.

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False) # We will provide OCR
model = AutoModelForTokenClassification.from_pretrained("microsoft/layoutlmv3-base")

print("LayoutLMv3 processor and model loaded.")

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LayoutLMv3 processor and model loaded.


In [ ]:
# Unzip the provided file
#!unzip /content/Synthetic_Motor_Claims_500.zip

In [5]:
import pandas as pd
import json
import os

# Define the base directory where the unzipped data is located
data_dir = r'C:\ArunzFolder\Arunzlab\CLAIM-CMEV\data\raw\synthetic_motor_claims_500'

# Load claims.jsonl (structured ground truth)
claims_path = os.path.join(data_dir, 'claims.jsonl')
claims_data = []
with open(claims_path, 'r') as f:
    for line in f:
        claims_data.append(json.loads(line))
claims_df = pd.DataFrame(claims_data)

print(f"Loaded {len(claims_df)} claims from {claims_path}")
print("\nClaims Data Head:")
display(claims_df.head())

# Load manifest.csv (image inventory and splits)
manifest_path = os.path.join(data_dir, 'manifest.csv')
manifest_df = pd.read_csv(manifest_path)

print(f"\nLoaded {len(manifest_df)} image manifest entries from {manifest_path}")
print("\nManifest Data Head:")
display(manifest_df.head())

# Load annotations.jsonl (image-level OCR regions and bounding boxes)
annotations_path = os.path.join(data_dir, 'annotations.jsonl')
annotations_data = []
with open(annotations_path, 'r') as f:
    for line in f:
        annotations_data.append(json.loads(line))
annotations_df = pd.DataFrame(annotations_data)

print(f"\nLoaded {len(annotations_df)} annotations from {annotations_path}")
print("\nAnnotations Data Head:")
display(annotations_df.head())

Loaded 250 claims from C:\ArunzFolder\Arunzlab\CLAIM-CMEV\data\raw\synthetic_motor_claims_500\claims.jsonl

Claims Data Head:


,claim_id,claim_number,claim_type,policy_excess,accident_date,assignment_date,survey_date,repairer,surveyor,vehicle,damage_description,parts,costs,split,images
0,SYN-00001,MC2026000001,Own Damage,750.0,2026-07-11,2026-07-12,2026-07-16,Progressive Auto Care,S. Kumar,"{'registration_number': 'SLF3508F', 'make': 'B...",Rear components damaged.,"[{'part_code': 'LIGHT-REAR-LAMP-LH', 'part_nam...","{'parts_subtotal': 1590.0, 'labour_subtotal': ...",train,"[images/train/SYN-00001_P1.jpg, images/train/S..."
1,SYN-00002,MC2026000002,Own Damage,750.0,2026-02-24,2026-02-25,2026-03-07,Eastside Accident Repair,Cheryl Ong,"{'registration_number': 'SRB1827G', 'make': 'B...","Front, Left, Right components damaged.","[{'part_code': 'MIRROR-DOOR-RH', 'part_name': ...","{'parts_subtotal': 3230.0, 'labour_subtotal': ...",train,"[images/train/SYN-00002_P1.jpg, images/train/S..."
2,SYN-00003,MC2025000003,Third Party,400.0,2025-11-23,2025-11-24,2025-12-03,Unity Motor Engineering,Wei Ming,"{'registration_number': 'SHD9346K', 'make': 'H...","Body, Front, Left, Rear, Right components dama...","[{'part_code': 'BODY-REAR-DOOR-RH', 'part_name...","{'parts_subtotal': 3120.0, 'labour_subtotal': ...",train,"[images/train/SYN-00003_P1.jpg, images/train/S..."
3,SYN-00004,MC2024000004,Own Damage,400.0,2024-05-27,2024-05-28,2024-06-06,Progressive Auto Care,S. Kumar,"{'registration_number': 'SZL427B', 'make': 'Te...","Front, Rear components damaged.",[{'part_code': 'BODY-FRONT-BUMPER-REINFORCEMEN...,"{'parts_subtotal': 3105.0, 'labour_subtotal': ...",train,"[images/train/SYN-00004_P1.jpg, images/train/S..."
4,SYN-00005,MC2024000005,Own Damage,500.0,2024-10-21,2024-10-22,2024-10-29,Central Auto Services,N. Ismail,"{'registration_number': 'SYA7709F', 'make': 'H...","Body, Front, Left, Rear, Right components dama...","[{'part_code': 'TRIM-BUMPER-RETAINER-RH', 'par...","{'parts_subtotal': 4395.0, 'labour_subtotal': ...",train,"[images/train/SYN-00005_P1.jpg, images/train/S..."



Loaded 500 image manifest entries from C:\ArunzFolder\Arunzlab\CLAIM-CMEV\data\raw\synthetic_motor_claims_500\manifest.csv

Manifest Data Head:


,image_id,claim_id,page_type,split,file_name,width,height
0,SYN-00001_P1,SYN-00001,survey_assessment,train,images/train/SYN-00001_P1.jpg,1240,1754
1,SYN-00001_P2,SYN-00001,repair_estimate,train,images/train/SYN-00001_P2.jpg,1240,1754
2,SYN-00002_P1,SYN-00002,survey_assessment,train,images/train/SYN-00002_P1.jpg,1240,1754
3,SYN-00002_P2,SYN-00002,repair_estimate,train,images/train/SYN-00002_P2.jpg,1240,1754
4,SYN-00003_P1,SYN-00003,survey_assessment,train,images/train/SYN-00003_P1.jpg,1240,1754



Loaded 500 annotations from C:\ArunzFolder\Arunzlab\CLAIM-CMEV\data\raw\synthetic_motor_claims_500\annotations.jsonl

Annotations Data Head:


,image_id,claim_id,page_number,page_type,split,file_name,width,height,regions
0,SYN-00001_P1,SYN-00001,1,survey_assessment,train,images/train/SYN-00001_P1.jpg,1240,1754,"[{'label': 'claim_number', 'text': 'MC20260000..."
1,SYN-00001_P2,SYN-00001,2,repair_estimate,train,images/train/SYN-00001_P2.jpg,1240,1754,"[{'label': 'claim_number', 'text': 'MC20260000..."
2,SYN-00002_P1,SYN-00002,1,survey_assessment,train,images/train/SYN-00002_P1.jpg,1240,1754,"[{'label': 'claim_number', 'text': 'MC20260000..."
3,SYN-00002_P2,SYN-00002,2,repair_estimate,train,images/train/SYN-00002_P2.jpg,1240,1754,"[{'label': 'claim_number', 'text': 'MC20260000..."
4,SYN-00003_P1,SYN-00003,1,survey_assessment,train,images/train/SYN-00003_P1.jpg,1240,1754,"[{'label': 'claim_number', 'text': 'MC20250000..."


In [6]:
from datasets import Dataset, Features, Value, ClassLabel, Array2D, Array3D
from datasets import Features, Sequence
from PIL import Image
import numpy as np

# --- 1. Define Labels and Mappings ---
# Collect all unique labels from the annotations
all_labels = []
for record in annotations_data:
    for region in record['regions']:
        all_labels.append(region['label'])

unique_labels = sorted(list(set(all_labels)))

# Define ID-to-label and label-to-ID mappings for IOB2 format
# 'O' for outside any entity
# 'B-' for beginning of an entity
# 'I-' for inside an entity
labels = ['O']
for label in unique_labels:
    labels.append(f"B-{label}")
    labels.append(f"I-{label}")

id2label = {str(i): label for i, label in enumerate(labels)}
label2id = {label: str(i) for i, label in enumerate(labels)}

print("Unique Labels Found:", unique_labels)
print("IOB2 Labels:", labels)
print("Label2ID Mapping:", label2id)

# --- 2. Prepare data for Dataset creation ---
# We need to combine information from manifest_df and annotations_df
# Create a dictionary to quickly lookup annotations by image_id
annotations_by_image_id = {record['image_id']: record for record in annotations_data}

processed_data = []
for idx, row in manifest_df.iterrows():
    image_id = row['image_id']
    image_file_path = os.path.join(data_dir, row['file_name'])
    annotations = annotations_by_image_id.get(image_id)

    if annotations and os.path.exists(image_file_path):
        # Extract words, boxes, and labels from annotations
        words = [region['text'] for region in annotations['regions']]
        bboxes = [region['bbox'] for region in annotations['regions']]
        # Map original labels to IOB2 format
        # For simplicity in this initial step, we assume each region is a single token,
        # and thus only use B- prefix. More robust tokenization (e.g. subword) would
        # require careful IOB2 tagging across tokens of a single region.
        word_labels = [label2id[f"B-{region['label']}"] for region in annotations['regions']]

        processed_data.append({
            'id': image_id,
            'image_path': image_file_path,
            'words': words,
            'bboxes': bboxes,
            'word_labels': word_labels, # These are already in ID form
            'image_width': annotations['width'],
            'image_height': annotations['height']
        })

# Convert to Hugging Face Dataset format
# Define the features of the dataset
features = Features({
    'id': Value(dtype='string'),
    'image_path': Value(dtype='string'),
    'words': Sequence(Value(dtype='string')),
    'bboxes': Sequence(Sequence(Value(dtype='int64'))), # Each bbox is [x_min, y_min, x_max, y_max]
    'word_labels': Sequence(ClassLabel(names=labels)),
    'image_width': Value(dtype='int64'),
    'image_height': Value(dtype='int64')
})

# Create the dataset
dataset = Dataset.from_list(processed_data, features=features)

# Split the dataset into train, validation, and test based on manifest
train_dataset = dataset.filter(lambda example: manifest_df[manifest_df['image_id'] == example['id']]['split'].iloc[0] == 'train')
val_dataset = dataset.filter(lambda example: manifest_df[manifest_df['image_id'] == example['id']]['split'].iloc[0] == 'validation')
test_dataset = dataset.filter(lambda example: manifest_df[manifest_df['image_id'] == example['id']]['split'].iloc[0] == 'test')


print(f"\nTotal processed samples: {len(dataset)}")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

print("\nExample from train_dataset:")
print(train_dataset[0])

print("\nDataset preparation complete. Next, we will set up the model and trainer.")

Unique Labels Found: ['accident_date', 'assignment_date', 'brakes', 'capacity', 'chassis_or_vin', 'claim_number', 'claim_type', 'colour', 'damage_description', 'date_of_accident', 'engine_or_motor_number', 'excess', 'general_condition', 'gst', 'gst_9', 'inspection_type', 'labour_subtotal', 'make_and_model', 'odometer_km', 'other_charges', 'part_1_action', 'part_1_amount', 'part_1_description', 'part_1_quantity', 'part_1_remarks', 'part_2_action', 'part_2_amount', 'part_2_description', 'part_2_quantity', 'part_2_remarks', 'part_3_action', 'part_3_amount', 'part_3_description', 'part_3_quantity', 'part_3_remarks', 'part_4_action', 'part_4_amount', 'part_4_description', 'part_4_quantity', 'part_4_remarks', 'part_5_action', 'part_5_amount', 'part_5_description', 'part_5_quantity', 'part_5_remarks', 'part_6_action', 'part_6_amount', 'part_6_description', 'part_6_quantity', 'part_6_remarks', 'part_7_action', 'part_7_amount', 'part_7_description', 'part_7_quantity', 'part_7_remarks', 'part_8_

Filter: 100%|██████████| 500/500 [00:00<00:00, 2003.52 examples/s]


Total processed samples: 500
Train samples: 350
Validation samples: 76
Test samples: 74

Example from train_dataset:
{'id': 'SYN-00001_P1', 'image_path': 'C:\\ArunzFolder\\Arunzlab\\CLAIM-CMEV\\data\\raw\\synthetic_motor_claims_500\\images/train/SYN-00001_P1.jpg', 'words': ['MC2026000001', '2026-07-11', 'Own Damage', '2026-07-12', '750.00', '2026-07-16', 'Progressive Auto Care', 'S. Kumar', 'SLF3508F', '12,282', 'BMW 320i', 'Brown', 'RGSLZT4AAZ9DRSA74', '2021-12-26', 'ENG8201806', '1998 cc', 'Good', 'Stationary', 'Not affected', 'Serviceable', '195/65 R15', '6 mm', 'Rear components damaged.', '2,147.30', '750.00', '14', '177.30'], 'bboxes': [[322, 222, 477, 238], [862, 219, 972, 237], [323, 272, 482, 297], [862, 265, 993, 285], [318, 321, 393, 340], [862, 320, 979, 338], [322, 377, 500, 397], [860, 371, 959, 388], [323, 490, 434, 507], [859, 482, 936, 505], [320, 533, 451, 554], [859, 540, 916, 555], [320, 586, 576, 605], [862, 592, 998, 611], [322, 643, 474, 662], [861, 639, 960, 658

In [13]:
from transformers import AutoProcessor, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Features, Value, ClassLabel, Sequence, Array2D
from PIL import Image
import torch

# --- 3. Initialize Processor and Model for Fine-tuning ---
# Reload processor and model with our custom label mappings
processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# Bypass cuDNN's convolution engine selection
torch.backends.cudnn.enabled = True

print("LayoutLMv3 processor and model initialized for token classification with custom labels.")

# --- 4. Define Data Collator ---
# We need a custom collator to handle variable-length sequences and pad them.
# The processor handles tokenization and alignment of words/boxes.
def collate_fn(batch):
    images = []
    words = []
    boxes = [] # This will now store normalized and clipped bounding boxes
    word_labels = []

    for item in batch:
        # Load image from path and convert to RGB
        img = Image.open(item['image_path']).convert("RGB")
        images.append(img)
        words.append(item['words'])

        # Get original image dimensions
        img_width, img_height = img.size

        # Normalize bounding boxes to 0-999 range and then clip
        normalized_bboxes_for_image = []
        for bbox_list_for_word in item['bboxes']:
            # Ensure coordinates are within [0, 999] range
            normalized_bbox = [
                max(0, min(999, int(1000 * (bbox_list_for_word[0] / img_width)))),
                max(0, min(999, int(1000 * (bbox_list_for_word[1] / img_height)))),
                max(0, min(999, int(1000 * (bbox_list_for_word[2] / img_width)))),
                max(0, min(999, int(1000 * (bbox_list_for_word[3] / img_height))))
            ]
            normalized_bboxes_for_image.append(normalized_bbox)
        boxes.append(normalized_bboxes_for_image)

        word_labels.append([int(label) for label in item['word_labels']])

    # The processor handles the full encoding and padding for the entire batch
    # Since we pre-normalized the bboxes, they are already in the 0-999 range.
    # The processor's internal normalization should then operate on these safe values.
    encoded_inputs = processor(
        images,
        words,
        boxes=boxes, # These are now pre-normalized and clipped
        word_labels=word_labels, # These are the token labels for each word/subword
        return_tensors="pt",
        padding="max_length", # Pad to the maximum sequence length in the batch
        truncation=True # Truncate sequences to the model's max input length
    )

    # Ensure labels are of type long
    if 'labels' in encoded_inputs and encoded_inputs['labels'].dtype != torch.long:
        encoded_inputs['labels'] = encoded_inputs['labels'].to(torch.long)
    # Bbox should typically be long, processor usually handles this
    if 'bbox' in encoded_inputs and encoded_inputs['bbox'].dtype != torch.long:
        encoded_inputs['bbox'] = encoded_inputs['bbox'].to(torch.long)

    # Debugging: Check generated labels just before returning
    #if 'labels' in encoded_inputs:
    #    print(f"DEBUG collate_fn: Labels shape: {encoded_inputs['labels'].shape}")
    #    print(f"DEBUG collate_fn: Labels unique values: {torch.unique(encoded_inputs['labels'])}")
    #    print(f"DEBUG collate_fn: Labels min value: {encoded_inputs['labels'].min()}")
    #    print(f"DEBUG collate_fn: Labels max value: {encoded_inputs['labels'].max()}")
    #    print(f"DEBUG collate_fn: Model num_labels: {model.config.num_labels}")
    #    if encoded_inputs['labels'].max() >= model.config.num_labels and encoded_inputs['labels'].max() != -100:
    #        print("DEBUG collate_fn ERROR: Label value detected >= num_labels! This is likely the cause of CUDA assert.")
    #    if encoded_inputs['labels'].min() < -100:
    #        print("DEBUG collate_fn ERROR: Label value detected < -100! This is likely the cause of CUDA assert.")

    return encoded_inputs

print("Data collator defined.")

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LayoutLMv3 processor and model initialized for token classification with custom labels.
Data collator defined.


In [14]:
#!pip install --upgrade setuptools
#!pip install evaluate
#!pip install build wheel
#!pip install scikit-learn
#!pip install --upgrade transformers

import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from transformers import TrainingArguments, Trainer # Added TrainingArguments and Trainer import here

# --- 5. Define Evaluation Metrics ---
# Function to compute metrics for evaluation using scikit-learn
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Debugging: Check predictions and labels before converting to strings
    #print(f"DEBUG compute_metrics: Raw predictions shape: {predictions.shape}")
    #print(f"DEBUG compute_metrics: Raw labels shape: {labels.shape}")
    #print(f"DEBUG compute_metrics: Raw predictions min: {np.min(predictions[predictions != -100]) if np.any(predictions != -100) else 'N/A'}")
    #print(f"DEBUG compute_metrics: Raw predictions max: {np.max(predictions[predictions != -100]) if np.any(predictions != -100) else 'N/A'}")
    #print(f"DEBUG compute_metrics: Raw labels min: {np.min(labels[labels != -100]) if np.any(labels != -100) else 'N/A'}")
    #print(f"DEBUG compute_metrics: Raw labels max: {np.max(labels[labels != -100]) if np.any(labels != -100) else 'N/A'}")
    #print(f"DEBUG compute_metrics: Model num_labels: {model.config.num_labels}")


    # Remove ignored index (if -100) and convert to labels
    true_labels = []
    for label_seq in labels:
        current_labels = []
        for l in label_seq:
            if l != -100:
                if str(l) in id2label:
                    current_labels.append(id2label[str(l)])
                else:
                    print(f"DEBUG compute_metrics ERROR: Label ID {l} not found in id2label!")
                    # Potentially replace with a default or skip, depending on desired behavior
        true_labels.append(current_labels)

    true_predictions = []
    for pred_seq, label_seq in zip(predictions, labels):
        current_predictions = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100: # Only consider predictions where ground truth label is not ignored
                if str(p) in id2label:
                    current_predictions.append(id2label[str(p)])
                else:
                    print(f"DEBUG compute_metrics ERROR: Prediction ID {p} not found in id2label!")
                    # Potentially replace with a default or skip
        true_predictions.append(current_predictions)


    # Flatten lists for scikit-learn metrics
    flat_true_labels = [item for sublist in true_labels for item in sublist]
    flat_true_predictions = [item for sublist in true_predictions for item in sublist]

    if not flat_true_labels: # Handle cases where no valid labels are present
        print("DEBUG compute_metrics: No valid labels found for metric computation.")
        return {"f1": 0.0, "precision": 0.0, "recall": 0.0, "accuracy": 0.0}

    # Calculate metrics using scikit-learn. Use 'macro' average for multi-class classification
    # Added error handling for empty lists after filtering - although `if not flat_true_labels` covers it
    if not flat_true_predictions:
        print("DEBUG compute_metrics: No valid predictions found for metric computation.")
        return {"f1": 0.0, "precision": 0.0, "recall": 0.0, "accuracy": 0.0}

    f1 = f1_score(flat_true_labels, flat_true_predictions, average='macro', zero_division=0)
    precision = precision_score(flat_true_labels, flat_true_predictions, average='macro', zero_division=0)
    recall = recall_score(flat_true_labels, flat_true_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(flat_true_labels, flat_true_predictions)

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy
    }

# --- 6. Configure Training Arguments ---
training_args = TrainingArguments(
    output_dir="./layoutlmv3_finetuned_claims",
    max_steps=1000, # Reduced for demonstration, consider more steps for better performance
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    # logging_dir="./logs", # Temporarily removed due to TypeError
    logging_steps=50,
    save_steps=500,
    seed=42,
    fp16=True, # Temporarily enabled for debugging CUDA error
    learning_rate=5e-5,
    # evaluation_strategy="steps", # Temporarily removed due to TypeError
    # eval_steps=50, # Temporarily removed due to TypeError
    # save_strategy="steps", # Temporarily removed due to TypeError
    load_best_model_at_end=False, # Set to False since evaluation strategy is removed
    # metric_for_best_model="f1", # Temporarily removed as load_best_model_at_end is False
    report_to="none", # Disable integration with experiment tracking tools for simplicity
    remove_unused_columns=False # Keep columns like 'words', 'bboxes', 'word_labels' needed by collate_fn
)

# Clear CUDA cache before Trainer initialization
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- 7. Initialize the Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    # tokenizer=processor, # Removed as Trainer does not accept this argument directly
    compute_metrics=compute_metrics
)

print("Training arguments and Trainer initialized.")

max_steps is given, it will override any value given in num_train_epochs


Training arguments and Trainer initialized.


In [ ]:
#from transformers import AutoProcessor, AutoModelForTokenClassification, TrainingArguments, Trainer
#from datasets import Features, Value, ClassLabel, Sequence, Array2D
#from PIL import Image
#import torch

# Bypass cuDNN's convolution engine selection
#torch.backends.cudnn.enabled = True

# --- 3. Initialize Processor and Model for Fine-tuning ---
# Reload processor and model with our custom label mappings
#processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
#model = AutoModelForTokenClassification.from_pretrained(
#    "microsoft/layoutlmv3-base",
#    num_labels=len(labels),
#    id2label=id2label,
#    label2id=label2id
#)

#print("LayoutLMv3 processor and model initialized for token classification with custom labels.")

# --- 4. Define Data Collator ---
# We need a custom collator to handle variable-length sequences and pad them.
# The processor handles tokenization and alignment of words/boxes.
#def collate_fn(batch):
#    images = []
#    words = []
#    boxes = []
#    word_labels = []

#    for item in batch:
#        # Load image from path and convert to RGB
#        img = Image.open(item['image_path']).convert("RGB")
#        images.append(img)
#        words.append(item['words'])
#        boxes.append(item['bboxes'])
        # Convert word_labels from string IDs back to integer IDs for processing if necessary
        # Assuming word_labels are already in the correct format (list of ints) from dataset creation
#        word_labels.append([int(label) for label in item['word_labels']])

    # The processor handles the full encoding and padding for the entire batch
    # It will return 'input_ids', 'attention_mask', 'bbox', 'pixel_values', 'labels'
#    encoded_inputs = processor(
#        images,
#        words,
#        boxes=boxes,
#        word_labels=word_labels, # These are the token labels for each word/subword
#        return_tensors="pt",
#        padding="max_length", # Pad to the maximum sequence length in the batch
#        truncation=True # Truncate sequences to the model's max input length
#    )

    # Ensure labels are of type long
#    if 'labels' in encoded_inputs and encoded_inputs['labels'].dtype != torch.long:
#        encoded_inputs['labels'] = encoded_inputs['labels'].to(torch.long)
#    # Bbox should typically be long, processor usually handles this
#    if 'bbox' in encoded_inputs and encoded_inputs['bbox'].dtype != torch.long:
#        encoded_inputs['bbox'] = encoded_inputs['bbox'].to(torch.long)

#    return encoded_inputs

#print("Data collator defined.")

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LayoutLMv3 processor and model initialized for token classification with custom labels.
Data collator defined.


In [15]:
print("Batch size:", trainer.args.per_device_train_batch_size)
print("Checkpointing:", trainer.args.gradient_checkpointing)
print("FP16:", trainer.args.fp16)

Batch size: 1
Checkpointing: False
FP16: True


In [16]:
# --- 8. Start Training ---

for name in [
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "gradient_checkpointing",
    "fp16",
    "max_steps",
]:
    print(name, getattr(trainer.args, name))
    
trainer.train()

print("\nModel training complete!")
print("Next, we will evaluate the fine-tuned model on the test set.")

per_device_train_batch_size 1
gradient_accumulation_steps 4
gradient_checkpointing False
fp16 True
max_steps 1000


  0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\arunz\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
  5%|▌         | 50/1000 [01:58<37:33,  2.37s/it]

{'loss': 3.3838, 'grad_norm': 10.09302806854248, 'learning_rate': 4.755e-05, 'epoch': 0.57}


 10%|█         | 100/1000 [03:58<35:46,  2.39s/it]

{'loss': 1.2447, 'grad_norm': 5.4146809577941895, 'learning_rate': 4.5050000000000004e-05, 'epoch': 1.14}


...

{'loss': 0.0031, 'grad_norm': 0.04057798907160759, 'learning_rate': 2.55e-06, 'epoch': 10.86}


100%|██████████| 1000/1000 [40:08<00:00,  2.41s/it]

{'loss': 0.003, 'grad_norm': 0.04057743400335312, 'learning_rate': 5.0000000000000004e-08, 'epoch': 11.43}


100%|██████████| 1000/1000 [40:11<00:00,  2.41s/it]

{'train_runtime': 2411.7774, 'train_samples_per_second': 1.659, 'train_steps_per_second': 0.415, 'train_loss': 0.26350691543519494, 'epoch': 11.43}

Model training complete!
Next, we will evaluate the fine-tuned model on the test set.


## 9. Evaluate the Fine-tuned Model on the Test Set

In [17]:
print("Evaluating the model on the test dataset...")
eval_results = trainer.evaluate(test_dataset)
print("Evaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

Evaluating the model on the test dataset...


c:\Users\arunz\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
100%|██████████| 74/74 [00:17<00:00,  4.19it/s]

Evaluation Results:
  eval_loss: 0.0015
  eval_f1: 1.0000
  eval_precision: 1.0000
  eval_recall: 1.0000
  eval_accuracy: 1.0000
  eval_runtime: 18.3075
  eval_samples_per_second: 4.0420
  eval_steps_per_second: 4.0420
  epoch: 11.4286


## 10. Save the Fine-tuned Model and Processor

In [18]:
import os
from transformers import TrainingArguments

# Re-define TrainingArguments to get the output_dir, ensuring it's available
training_args = TrainingArguments(
    output_dir=r"C:\ArunzFolder\Arunzlab\CLAIM-CMEV\artifacts\models\document-worker-layoutlmv3",
)
model_output_dir = training_args.output_dir

# The 'model' and 'processor' objects are already defined and fine-tuned/initialized
# from previous cells (bc88f68d, c3dc20f2, 1930200d).
# We just need to save these in-memory objects.

print(f"Saving the fine-tuned model and processor to: {model_output_dir}...")
model.save_pretrained(model_output_dir)
processor.save_pretrained(model_output_dir)

print(f"Fine-tuned model and processor saved to: {model_output_dir}")
print("You can load them later using: `AutoModelForTokenClassification.from_pretrained(model_output_dir)` and `AutoProcessor.from_pretrained(model_output_dir)`")

Saving the fine-tuned model and processor to: C:\ArunzFolder\Arunzlab\CLAIM-CMEV\artifacts\models\document-worker-layoutlmv3...
Fine-tuned model and processor saved to: C:\ArunzFolder\Arunzlab\CLAIM-CMEV\artifacts\models\document-worker-layoutlmv3
You can load them later using: `AutoModelForTokenClassification.from_pretrained(model_output_dir)` and `AutoProcessor.from_pretrained(model_output_dir)`
